###🔥 Read Private data from XML file

In [0]:
df = spark.read \
    .format("xml") \
    .option("rowTag", "Transaction") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .load("/Volumes/demo_catalog/demo_schema/demo_raw/xml_file/")

df.display()

####👉 Filter data "like"

In [0]:
from pyspark.sql.functions import col

df_filtered = df.filter(
    col("TRANSACTION_TYPE_DESCRIPTION").like("Income:%")
)

df_filtered.display()

####👉 Filter string start with "startswith"

In [0]:
from pyspark.sql.functions import col

df_filtered = df.filter(
    col("TRANSACTION_TYPE_DESCRIPTION").startswith("Income:")
)

df_filtered.display()

####👉 Distinct column value

In [0]:
df_q = df.select("TRANSACTION_TYPE").distinct()
df_q.display()

In [0]:
df.printSchema()

In [0]:
GPA_NGP_PFPM_DBK
│
├── pipeline_package
│   ├── silver
│   ├── tests
│   └── utils
│       ├── argument_parser.py
│       ├── CommonUtilityFunctionsFactory.py
│       ├── compute_xirr_security.py
│       ├── ConfigFilePath.py
│       ├── DateUtilsFactory.py
│       ├── EnvironmentConfigLoader.py
│       ├── Filefactory.py
│       ├── GoldConsumptionConstants.py
│       ├── GoldPBORConstants.py   ← (open in editor)
│       ├── GoldPrdConstants.py
│       ├── MetadataConstants.py
│       ├── PFPM_common_utility_functions.py
│       ├── PFPM_common_utility_variables.py
│       ├── quarterly_qtr.py
│       ├── SFConnectorFactory.py
│       ├── SilverConstants.py
│       ├── xirr_calculation.py
│       ├── XIRRCalculatorSecurity.py
│       ├── writer.py
│       └── __init__.py


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window
from datetime import datetime
import logging

# ------ need to remove
import sys
sys.path.append('/Workspace/Shared/PrivateMarket_V3/Data_Loads/utils/')
sys.path.append('/Workspace/Shared/PrivateMarket_V3/Data_Loads/Logging/')
sys.path.append('/Workspace/Shared/PrivateMarket_V3/Data_Loads/Factory/')
# ------ need to remove

from PFPH_common_utility_functions import get_transaction_type_list
from CommonUtilityFunctionsFactory import CommonUtilityFunctionsFactory
from setup_logging import setup_logging
from DataReaderFactory import DataReaderFactory
from DataWriterFactory import DataWriterFactory
from EnvironmentConfigLoader import EnvironmentConfigLoader
from GoldPBORConstants import GoldPBORConstants


In [0]:
GOLD_PBOR_ENTITY_DAILY = "entity_calculation_daily"
GOLD_PBOR_ENTITY_DAILY_INSERT_KEYS = {
    "_HASH_ENTITY_INVESTOR_QTR": ["HASH_INVESTOR_ID", "HASH_ENTITY_ID"],
    "CONTRIBUTION": "DISTRIBUTION", 
    "PROCESS_DATE": ""
}

PBOR_FUND_CALCULATION_QTR_TBL = "entity_calculation_qtr"
PBOR_STG_FUND_SUMMARY_TBL = "stage_fund_calculation_base"
PBOR_STG_FUND_XIRR_BASE = "stage_fund_xirr_base"
PBOR_STG_XIRR_QTR_TBL = "stg_fund_xirr_qtr"
PBOR_STG_ADJUSTED_SECURITY_TBL = "stg_adjusted_security"
PBOR_SECURITY_CALCULATION_BASE = "stage_security_calculation_base"
PBOR_SECURITY_XIRR_BASE = "stage_security_xirr_base"
PBOR_SECURITY_XIRR_QTR_TBL = "stage_security_xirr_qtr"
PBOR_SECURITY_CALCULATION_QTR_TBL = "entity_calculation_qtr"
PBOR_SECURITY_GROUP_COLS = ["HASH_ENTITY_ID", "HASH_INVESTOR_ID"]
PBOR_SECURITY_CALCULATION_QTR_MERGE_KEY = ["HASH_ENTITY_ID", "HASH_INVESTOR_ID"]


In [0]:
"CONTRIBUTION_YR",
when(
    quarter(fund["QTR_END_DATE"]).isin([1, 2, 3]),
).otherwise(
    pyspark_sum(stg_fund["QTR_CONTRIBUTION"]).over(fund_window_spec)
)).withColumn(
    "DISTRIBUTION_YR",
when(
    quarter(fund["QTR_END_DATE"]).isin([1, 2, 3]),
).otherwise(
    pyspark_sum(stg_fund["QTR_DISTRIBUTION"]).over(fund_window_spec)
)).withColumn("TOTAL_VALUE", lit(0)) \
.withColumn("REALISED_VALUE", lit(0)) \
.withColumn("UNREALISED_VALUE", lit(0)) \
.withColumn(
    "PREVIOUS_IRR",
    lag(fund["IRR"]).over(fund_irr_window_spec)
)
#-----------
df_entity_master = (
    
    source_reader.read(spark, f"{const_dict['pm_medallion_catalog']}")
    .filter(col("ENTITY_ID").isin([row["ENTITY_ID"] for row in ...]))
    .select(
        "ENTITY_ID",
        "CURRENCY_CODE_BASE_LEGAL_ENT",
        "ENTITY FAMILY NAME"
    )
    .withColumn("PARENT_ID", lit(None))
    .withColumn("ENTITY_TYPE", lit("ENTITY"))
    .dropDuplicates(["ENTITY_ID", "CURRENCY_CODE_BASE_LEGAL_ENT"])
)

df_gl_investment = (
    source_reader.read(spark, f"{const_dict['pm_medallion_catalog']}")
    .filter(col("ENTITY_ID").isin([row["ENTITY_ID"] for row in ...]))
    .select(
        col("DEAL_ID").alias("ENTITY_ID")
    )
)

# Joining the gold_entity_transaction_tbl, gold_entity_master_tbl, gold_dim_source_master_tbl to get the Source_ID
entity_transaction_df.alias("investment") \
    .join(entity_master_df.alias("fund"), f.col("investment_HASH_ENTITY_ID") == f.col("fund_HASH_ENTITY_ID"), "left") \
    .join(source_df.alias("source"), f.col("fund_HASH_SOURCE_ID") == f.col("source_HASH_SOURCE_ID"), "left") \
    .join(date_df.alias("dt"), f.col("investment_TRANSACTION_DATEKEY") == f.col("dt_date_key"), "left") \
    .filter(
        f.lower(f.col("investment_TRANSACTION_TYPE_DESCRIPTION")).isin(
            [f.lit(x.lower()) for x in nav_transaction_type + contrl_transaction_type + distrl_transaction_type]
        )
    ) \
    .filter(f.col("investment_HASH_ENTITY_ID").isin(entity_id_list)) \
    .select(
        f.col("source_SOURCE_ID"),
        f.col("investment_HASH_ENTITY_ID"),
        f.col("fund_ENTITY_ID"),
        f.col("investment_TRANSACTION_AMOUNT"),
        f.col("investment_TRANSACTION_TYPE_DESCRIPTION"),
        f.col("investment_TRANSACTION_DATEKEY"),
        f.col("dt_date").alias("transaction_date"),
        f.col("dt_year"),
        f.col("dt_quarter"),
        f.col("investment_UPDATED_DATETIME")
    )


In [0]:
git:gpa_ngp_pfpm_dbx / pipeline_package / gold_consumption / load_gold_consumption_summary_qtr.py


116     .otherwise(
117         pyspark_sum(fund["IRR"]).over(yearly_irr_window_spec)
118     )
119     ) \
120     .select(
121         F.coalesce(dim_node["SK_NODE_ID"], lit(None)).alias("SK_NODE_ID"),
122         fund["QTR_END_DATE"].alias("QUARTER_END_DATE"),
123         F.col("AGE"),
124         fund["NAV"],
125         fund["INVESTED_CAPITAL"],
126         fund["CALLABLE_CAPITAL"],
127         F.coalesce(stg_fund["ENTITY_COMMITMENT_AMOUNT"], lit(0)).alias("COMMITMENT_AMOUNT"),
128         fund["CONTRIBUTION"].alias("CONTRIBUTION_QTR"),
129     fund['DISTRIBUTION'].alias('DISTRIBUTION_QTR'),
130     F.col('CONTRIBUTION_YR'),
131     F.col('DISTRIBUTION_YR'),
132     lit(0).alias('COST'),
133     F.col('TOTAL_VALUE'),
134     F.col('REALISED_VALUE'),
135     F.col('UNREALISED_VALUE'),
136     fund['TVPI'],
137     fund['DVPI'].alias('DPI'),
138     fund['MOIC'],
139     fund['IRR'].alias('IRR_QTR'),
140     F.col('IRR_YR'),
141     F.col('IRR_CHANGE_FROM_LAST_QTR'),
142     lit(0.00).alias('PERCENTAGE_OF_CURRENT_VALUE')
143     )

144 logging.info(f"Final result count: {source_entity_calc_qtr_df.count()}")

145 # Join security calculations with dim_entity first
146 security_with_entity = security.join(dim_entity, security["_HASH_ENTITY_ID"] == dim_entity["_HASH_ENTITY_ID"], "left")



In [0]:
# ------------ JOIN TABLES ------------
# Join fund calculations with entity dimension
fund_with_entity = fund.join(dim_entity, fund["_HASH_ENTITY_ID"] == dim_entity["_HASH_ENTITY_ID"], "left")
logging.info(f"After dim_entity join: {fund_with_entity.count()}")

# Join with node dimension to get hierarchy information
fund_with_node = fund_with_entity.join(dim_node, fund_with_entity["ENTITY_ID"] == dim_node["NODE_ID"], "left")

logging.info(f"After dim_node join: {fund_with_node.count()}")

# Join with staging fund data to get commitment amounts
fund_enriched = fund_with_node.join(stg_fund,
    (fund["_HASH_ENTITY_ID"] == stg_fund["_HASH_ENTITY_ID"]) &
    (fund["_HASH_INVESTOR_ID"] == stg_fund["_HASH_INVESTOR_ID"]) &
    (fund["QTR_END_DATE"] == stg_fund["TRANSACTION_DATE_QTR"]),
    "left")

logging.info(f"After stg_fund join: {fund_enriched.count()}")
##########------- 
"CONTRIBUTION_YR",
when(
    quarter(fund["QTR_END_DATE"]).isin([1, 2, 3]),
).otherwise(
    pyspark_sum(stg_fund["QTR_CONTRIBUTION"]).over(fund_window_spec)
)).withColumn(
    "DISTRIBUTION_YR",
when(
    quarter(fund["QTR_END_DATE"]).isin([1, 2, 3]),
).otherwise(
    pyspark_sum(stg_fund["QTR_DISTRIBUTION"]).over(fund_window_spec)
)).withColumn("TOTAL_VALUE", lit(0)) \
.withColumn("REALISED_VALUE", lit(0)) \
.withColumn("UNREALISED_VALUE", lit(0)) \
.withColumn(
    "PREVIOUS_IRR",
    lag(fund["IRR"]).over(fund_irr_window_spec)
)



In [0]:
%sql
-- select * from npee_gpangp_dev_catalog_01.gold.stg_security_xirr_qtr
-- where "_HASH_INVESTOR_ID" = '-8667385132751198733'
-- and "_HASH_SECURITY_ID" = '4058096794784778307'

select node.NODE_ID, node.MAST_TYPE_ID, node.NODE_NAME, smry.* 
from npee_gpangp_dev_catalog_01.gold.summary_qtr smry
join npee_gpangp_dev_catalog_01.gold.dim_node node
on smry.SK_NODE_ID = node.SK_NODE_ID
where node.NODE_ID in ('1-4488', '2-57')


In [0]:
%sql
select * from npe_gpangp_dev_catalog_01.gold_entity_calculation_qtr
where ENTITY_ID = '2778'
-- where HASH_ENTITY_ID = '-44442825017024778897'

select ENTITY_ID, SECURITY_ID, DEAL_NAME, INVESTOR_ID_SOURCE_SYSTEM, EFFECTIVE_DATE,
       TRANSACTION_ID, TRANSACTION_TYPE_DESCRIPTION, INVESTOR_COMMITMENT_AMOUNT,
       TRANSACTION_ALLOCATION_AMOUNT_BASE,
       count(*) as ROW_COUNT
from npe_gpangp_dev_catalog_01.silver.ios_general_ledger_activity_investor
where EFFECTIVE_DATE between '2023-01-01' and '2023-03-31'
  and ENTITY_ID = '2-57'


In [0]:
PBOR_FUND_CALCULATION_QTR_UPDATE_KEY = [
    "CALLABLE_CAPITAL", "CONTRIBUTION", "DISTRIBUTION",
    "IRR", "INVESTED_CAPITAL", "PROCESS_DATE"
]

PBOR_FUND_CALCULATION_QTR_INSERT_KEY = [
    "_HASH_ENTITY_INVESTOR_QTR", "_HASH_ENTITY_ID",
    "CONTRIBUTION", "DISTRIBUTION", "NAV", "TVPI", "DVPI", "RVPI", "MOIC"
]

GOLD_PBOR_ENTITY_DAILY = "entity_calculation_daily"
GOLD_PBOR_ENTITY_DAILY_MERGE_KEYS = [
    "_HASH_ENTITY_INVESTOR_DAILY", "_HASH_ENTITY_ID"
]

GOLD_PBOR_ENTITY_DAILY_UPDATE_KEYS = ["CONTRIBUTION", "DISTRIBUTION"]

GOLD_PBOR_ENTITY_DAILY_INSERT_KEYS = [
    "_HASH_ENTITY_INVESTOR_DAILY", "_HASH_ENTITY_ID",
    "CONTRIBUTION", "DISTRIBUTION", "PROCESS_DATE"
]

PBOR_FUND_CALCULATION_QTR_TBL = "entity_calculation_qtr"


# reading the data from source tables
entity_transaction_df = source_reader.read(
    spark, f"{const_dict['p_medallion_catalog_var']}:{const_dict['p_GOLD_SCHEMA_VAR']}.{const_dict['p_ENTITY_TRANSACTION_TBL']}"
)
entity_master_df = source_reader.read(
    spark, f"{const_dict['p_medallion_catalog_var']}:{const_dict['p_GOLD_SCHEMA_VAR']}.PBOR_FUND_CALCULATION_QTR_UPDATE_KEY = [
    "CALLABLE_CAPITAL", "CONTRIBUTION", "DISTRIBUTION",
    "IRR", "INVESTED_CAPITAL", "PROCESS_DATE"
]

PBOR_FUND_CALCULATION_QTR_INSERT_KEY = [
    "_HASH_ENTITY_INVESTOR_QTR", "_HASH_ENTITY_ID",
    "CONTRIBUTION", "DISTRIBUTION", "NAV", "TVPI", "DVPI", "RVPI", "MOIC"
]

GOLD_PBOR_ENTITY_DAILY = "entity_calculation_daily"
GOLD_PBOR_ENTITY_DAILY_MERGE_KEYS = [
    "_HASH_ENTITY_INVESTOR_DAILY", "_HASH_ENTITY_ID"
]

GOLD_PBOR_ENTITY_DAILY_UPDATE_KEYS = ["CONTRIBUTION", "DISTRIBUTION"]

GOLD_PBOR_ENTITY_DAILY_INSERT_KEYS = [
    "_HASH_ENTITY_INVESTOR_DAILY", "_HASH_ENTITY_ID",
    "CONTRIBUTION", "DISTRIBUTION", "PROCESS_DATE"
]

PBOR_FUND_CALCULATION_QTR_TBL = "entity_calculation_qtr"
{const_dict['p_ENTITY_MASTER_TBL']}"


)
source_df = source_reader.read(
    spark, f"{const_dict['p_medallion_catalog_var']}:{const_dict['p_GOLD_SCHEMA_VAR']}.{const_dict['p_GOLD_DIX_SOURCE_MASTER_TBL']}"
)
fund_summary_df = source_reader.read(
    spark, f"{const_dict['p_medallion_catalog_var']}:{const_dict['p_GOLD_SCHEMA_VAR']}.{const_dict['p_FUND_SUMMARY_TBL']}"
)

# joining the gold_entity_transaction_tbl, gold_entity_master_tbl, gold_dix_source_master_tbl to get the source_ID
base_table_df = (
    entity_transaction_df.alias("investment")
    .join(entity_master_df.alias("fund"), f.col("investment.HASH_ENTITY_ID") == f.col("fund.HASH_ENTITY_ID"), "left")
    .join(source_df.alias("source"), f.col("fund.HASH_SOURCE_ID") == f.col("source.HASH_SOURCE_ID"), "left")
    .join(fund_summary_df.alias("summary"), f.col("fund.TRANSACTION_DATEKEY") == f.col("summary.data_key"), "left")
    .filter(
        f.lower(f.col("investment.TRANSACTION_TYPE_DESCRIPTION")).isin(
            [f.lit(x.lower()) for x in nav_transaction_type + contri_transaction_type + distri_transaction_type]
        )
    )
    .filter(f.col("investment.HASH_ENTITY_ID").isin(entity_id_list))
)


In [0]:
"spark_python_wheel_job": {
    "python_wheel_tasks": [
        {
            "task29": {
                "depends_on": {},
                "existing_cluster_name": "GPA_NGP_DEV_CLUSTER_6",
                "library_whl_paths": ["/Volumes/ssdssno_dev_catalog_01/app_1"],
                "python_wheel_task": {
                    "entry_point": "load_gold_consumption_summary_daily",
                    "package_name": "pipeline_package",
                    "parameters": [],
                    "named_parameters": {"ENVIRONMENT": "default"}
                }
            },
            "task30": {
                "task_key": "load_gold_consumption_summary_qtr",
                "run_if": "ALL_SUCCESS",
                "depends_on": {
                    "depends1": "load_gold_consumption_summary_daily"
                },
                "existing_cluster_name": "GPA_NGP_DEV_CLUSTER_6",
                "library_whl_paths": ["/Volumes/ssdssno_dev_catalog_01/app_1"],
                "python_wheel_task": {
                    "entry_point": "load_gold_consumption_summary_qtr",
                    "package_name": "pipeline_package"
                }
            }
        }
    ]
}
